# Data Merge and Analysis Notebook

This notebook merges and analyzes the following datasets:
- Age at Visit
- Demographics
- DaTScan SBR Analysis
- Data Dictionary (for reference)

The goal is to create a unified dataset for downstream analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
# Define data paths
data_dir = Path('data/csvData')

age_path = data_dir / 'Age_at_visit_14Oct2025.csv'
demographics_path = data_dir / 'Demographics_14Oct2025.csv'
datscan_path = data_dir / 'DaTScan_SBR_Analysis_17Dec2024.csv'
data_dict_path = data_dir / 'Data_Dictionary_-__Annotated__04Dec2025.csv'

# Load datasets
df_age = pd.read_csv(age_path)
df_demographics = pd.read_csv(demographics_path)
df_datscan = pd.read_csv(datscan_path)
df_data_dict = pd.read_csv(data_dict_path)

print("Data loaded successfully!")
print(f"Age at Visit: {df_age.shape}")
print(f"Demographics: {df_demographics.shape}")
print(f"DaTScan: {df_datscan.shape}")
print(f"Data Dictionary: {df_data_dict.shape}")

## 2. Explore Individual Datasets

In [ ]:
print("=" * 80)
print("AGE AT VISIT")
print("=" * 80)
print(df_age.head())
print(f"\nColumns: {df_age.columns.tolist()}")
print(f"Data types:\n{df_age.dtypes}")
print(f"\nMissing values:\n{df_age.isnull().sum()}")
print(f"\nBasic stats:\n{df_age.describe()}")

In [ ]:
print("=" * 80)
print("DEMOGRAPHICS")
print("=" * 80)
print(df_demographics.head())
print(f"\nColumns: {df_demographics.columns.tolist()}")
print(f"Data types:\n{df_demographics.dtypes}")
print(f"\nMissing values:\n{df_demographics.isnull().sum()}")

In [ ]:
print("=" * 80)
print("DATSCAN SBR ANALYSIS")
print("=" * 80)
print(df_datscan.head())
print(f"\nColumns: {df_datscan.columns.tolist()}")
print(f"Data types:\n{df_datscan.dtypes}")
print(f"\nMissing values:\n{df_datscan.isnull().sum()}")
print(f"\nBasic stats:\n{df_datscan.describe()}")

## 3. Identify Common Keys for Merging

In [ ]:
# Check common columns
print("Common columns across datasets:")
print(f"Age & Demographics: {set(df_age.columns) & set(df_demographics.columns)}")
print(f"Age & DaTScan: {set(df_age.columns) & set(df_datscan.columns)}")
print(f"Demographics & DaTScan: {set(df_demographics.columns) & set(df_datscan.columns)}")

# Check unique values
print(f"\nUnique PATNO in Age: {df_age['PATNO'].nunique()}")
print(f"Unique PATNO in Demographics: {df_demographics['PATNO'].nunique()}")
print(f"Unique PATNO in DaTScan: {df_datscan['PATNO'].nunique()}")

print(f"\nUnique EVENT_ID in Age: {df_age['EVENT_ID'].nunique()}")
print(f"Unique EVENT_ID in Demographics: {df_demographics['EVENT_ID'].nunique()}")
print(f"Unique EVENT_ID in DaTScan: {df_datscan['EVENT_ID'].nunique()}")

## 4. Data Cleaning and Preparation

In [ ]:
# Convert data types
df_age['PATNO'] = df_age['PATNO'].astype(int)
df_age['AGE_AT_VISIT'] = pd.to_numeric(df_age['AGE_AT_VISIT'], errors='coerce')

df_demographics['PATNO'] = df_demographics['PATNO'].astype(int)

df_datscan['PATNO'] = df_datscan['PATNO'].astype(int)

# Convert numeric columns in DaTScan
numeric_cols = ['DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L', 'DATSCAN_PUTAMEN_R', 
                'DATSCAN_PUTAMEN_L', 'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT']
for col in numeric_cols:
    df_datscan[col] = pd.to_numeric(df_datscan[col], errors='coerce')

print("Data types converted successfully!")

## 5. Merge Datasets

In [ ]:
# Merge Age with Demographics on PATNO and EVENT_ID
df_merged = pd.merge(
    df_age,
    df_demographics,
    on=['PATNO', 'EVENT_ID'],
    how='outer',
    indicator=True
)

print(f"After merging Age + Demographics: {df_merged.shape}")
print(f"Merge indicator:\n{df_merged['_merge'].value_counts()}")
df_merged = df_merged.drop('_merge', axis=1)

In [ ]:
# Merge with DaTScan
df_merged = pd.merge(
    df_merged,
    df_datscan,
    on=['PATNO', 'EVENT_ID'],
    how='outer',
    indicator=True
)

print(f"After merging with DaTScan: {df_merged.shape}")
print(f"Merge indicator:\n{df_merged['_merge'].value_counts()}")
df_merged = df_merged.drop('_merge', axis=1)

print(f"\nFinal merged dataset shape: {df_merged.shape}")
print(f"Columns: {df_merged.columns.tolist()}")

## 6. Merged Dataset Overview

In [ ]:
print("Merged Dataset Info:")
print(f"Shape: {df_merged.shape}")
print(f"\nFirst few rows:")
print(df_merged.head())

In [ ]:
# Missing values analysis
missing_pct = (df_merged.isnull().sum() / len(df_merged) * 100).sort_values(ascending=False)
print("Missing Values (%):\n")
print(missing_pct[missing_pct > 0])

In [ ]:
# Data types summary
print("Data Types:")
print(df_merged.dtypes)

## 7. Statistical Analysis

In [ ]:
# Patient statistics
print("Patient Statistics:")
print(f"Total unique patients: {df_merged['PATNO'].nunique()}")
print(f"Total records: {len(df_merged)}")
print(f"Average records per patient: {len(df_merged) / df_merged['PATNO'].nunique():.2f}")

# Event distribution
print(f"\nEvent ID distribution:")
print(df_merged['EVENT_ID'].value_counts())

In [ ]:
# Age statistics
print("Age at Visit Statistics:")
print(df_merged['AGE_AT_VISIT'].describe())

# Sex distribution
print(f"\nSex distribution:")
print(df_merged['SEX'].value_counts())

In [ ]:
# DaTScan metrics statistics
print("DaTScan Metrics Statistics:")
datscan_metrics = ['DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L', 'DATSCAN_PUTAMEN_R', 
                   'DATSCAN_PUTAMEN_L', 'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT']
print(df_merged[datscan_metrics].describe())

## 8. Visualizations

In [ ]:
# Age distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_merged['AGE_AT_VISIT'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age at Visit')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Age at Visit')
axes[0].grid(True, alpha=0.3)

# Sex distribution
sex_counts = df_merged['SEX'].value_counts()
axes[1].bar(['Female', 'Male'], [sex_counts.get(0, 0), sex_counts.get(1, 0)], color=['pink', 'lightblue'], edgecolor='black')
axes[1].set_ylabel('Count')
axes[1].set_title('Sex Distribution')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# DaTScan metrics by hemisphere
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

datscan_metrics = ['DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L', 'DATSCAN_PUTAMEN_R', 
                   'DATSCAN_PUTAMEN_L', 'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT']

for idx, metric in enumerate(datscan_metrics):
    axes[idx].hist(df_merged[metric].dropna(), bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_xlabel(metric.replace('DATSCAN_', ''))
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution of {metric.replace("DATSCAN_", "")}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis for DaTScan metrics
datscan_data = df_merged[datscan_metrics].dropna()
if len(datscan_data) > 0:
    corr_matrix = datscan_data.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix of DaTScan Metrics')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough DaTScan data for correlation analysis")

## 9. Save Merged Dataset

In [ ]:
# Save merged dataset
output_path = Path('output/merged_data.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_merged.to_csv(output_path, index=False)

print(f"Merged dataset saved to: {output_path}")
print(f"Shape: {df_merged.shape}")

## 10. Summary Report

In [ ]:
print("\n" + "="*80)
print("MERGE AND ANALYSIS SUMMARY")
print("="*80)

print(f"\nDataset Dimensions:")
print(f"  - Age at Visit: {df_age.shape}")
print(f"  - Demographics: {df_demographics.shape}")
print(f"  - DaTScan: {df_datscan.shape}")
print(f"  - Merged: {df_merged.shape}")

print(f"\nPatient Information:")
print(f"  - Total unique patients: {df_merged['PATNO'].nunique()}")
print(f"  - Total records: {len(df_merged)}")
print(f"  - Average records per patient: {len(df_merged) / df_merged['PATNO'].nunique():.2f}")

print(f"\nData Completeness:")
print(f"  - Records with Age: {df_merged['AGE_AT_VISIT'].notna().sum()} ({df_merged['AGE_AT_VISIT'].notna().sum()/len(df_merged)*100:.1f}%)")
print(f"  - Records with Demographics: {df_merged['SEX'].notna().sum()} ({df_merged['SEX'].notna().sum()/len(df_merged)*100:.1f}%)")
print(f"  - Records with DaTScan: {df_merged['DATSCAN_CAUDATE_R'].notna().sum()} ({df_merged['DATSCAN_CAUDATE_R'].notna().sum()/len(df_merged)*100:.1f}%)")

print(f"\nAge Statistics:")
print(f"  - Mean: {df_merged['AGE_AT_VISIT'].mean():.2f}")
print(f"  - Std: {df_merged['AGE_AT_VISIT'].std():.2f}")
print(f"  - Min: {df_merged['AGE_AT_VISIT'].min():.2f}")
print(f"  - Max: {df_merged['AGE_AT_VISIT'].max():.2f}")

print(f"\nOutput:")
print(f"  - Merged dataset saved to: output/merged_data.csv")
print("\n" + "="*80)